# Point-in-time vertical slice

This notebook queries normalized daily prices, SEC fundamentals, and FRED macro Parquet in one DuckDB session. It resolves `security_id` to `issuer_id` through the collector's configured universe, then joins only observations whose conservative `available_at` is known. No sample observations are fabricated. On a fresh checkout the catalog exposes typed empty views and explains what ingestion must run first.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from research import ResearchCatalog, load_security_mappings

candidate = Path(os.getenv("INVS_DATA_ROOT", "/data"))
DATA_ROOT = candidate if candidate.exists() else Path.cwd().parent.parent / "data"
config_candidate = Path(os.getenv("INVS_CONFIG", "/etc/invs/config.yaml"))
CONFIG_PATH = (
    config_candidate
    if config_candidate.exists()
    else Path.cwd().parent.parent / "config" / "config.example.yaml"
)
catalog = ResearchCatalog(DATA_ROOT).register()
configured_mappings = load_security_mappings(CONFIG_PATH)
pd.DataFrame([item.__dict__ for item in catalog.status()])

## Select a reproducible slice

Only security/issuer pairs declared together in the collector universe are eligible. `EXAMPLE_SECURITY_ID` can pin one configured pair; the issuer is never selected independently. The SEC concept and FRED series can also be pinned, otherwise deterministic values are selected from the mapped issuer's ingested data.

In [ ]:
def configured_or_first(variable, query, parameters=None):
    configured = os.getenv(variable)
    if configured:
        return configured
    row = catalog.connection.execute(query, parameters or {}).fetchone()
    return row[0] if row else None

available_mappings = catalog.available_mappings(configured_mappings)
requested_security_id = os.getenv("EXAMPLE_SECURITY_ID")
eligible = [
    item for item in available_mappings
    if requested_security_id is None or item.security_id == requested_security_id
]
mapping = sorted(eligible, key=lambda item: (item.security_id, item.issuer_id))[0] if eligible else None
fundamental_concept = (
    configured_or_first(
        "EXAMPLE_SEC_CONCEPT",
        "SELECT concept FROM fundamentals WHERE issuer_id = $issuer_id AND concept IS NOT NULL ORDER BY concept LIMIT 1",
        {"issuer_id": mapping.issuer_id},
    )
    if mapping else None
)
macro_source = os.getenv("EXAMPLE_MACRO_SOURCE") or "fred"
macro_series_id = configured_or_first(
    "EXAMPLE_FRED_SERIES",
    "SELECT series_id FROM macroeconomics WHERE source = $macro_source AND series_id IS NOT NULL ORDER BY series_id LIMIT 1",
    {"macro_source": macro_source},
)
decision_at = os.getenv("EXAMPLE_DECISION_AT") or pd.Timestamp.now(tz="UTC").isoformat()
selection = {
    "decision_at": decision_at,
    "security_id": mapping.security_id if mapping else None,
    "issuer_id": mapping.issuer_id if mapping else None,
    "fundamental_concept": fundamental_concept,
    "macro_source": macro_source,
    "macro_series_id": macro_series_id,
}
selection

In [ ]:
if any(value is None for value in selection.values()):
    missing = ", ".join(catalog.missing()) or "one or more populated datasets"
    print(
        "No joined observations yet. Run the price, SEC, and selected macro collector first. "
        f"Missing or empty inputs: {missing}. The empty result below is expected on first boot."
    )
    joined = pd.DataFrame()
else:
    joined = catalog.research_snapshot(
        decision_at=decision_at,
        mapping=mapping,
        fundamental_concept=fundamental_concept,
        macro_source=macro_source,
        macro_series_id=macro_series_id,
    )

joined.tail(20)

## Lookahead guard

These assertions document the temporal contract. They tolerate missing fundamentals or macro values, but every joined value must have been available no later than the price row's `known_at` timestamp.

In [ ]:
if not joined.empty:
    known_at = pd.to_datetime(joined["known_at"], utc=True)
    fundamental_available = pd.to_datetime(joined["fundamental_available_at"], utc=True)
    macro_available = pd.to_datetime(joined["macro_available_at"], utc=True)
    has_fundamental = fundamental_available.notna()
    has_macro = macro_available.notna()
    assert (fundamental_available[has_fundamental] <= known_at[has_fundamental]).all()
    assert (macro_available[has_macro] <= known_at[has_macro]).all()
    print(f"Validated {len(joined):,} price rows as known at {decision_at}.")
else:
    print("Lookahead checks skipped because normalized inputs have not been ingested yet.")